# mit-opere-incompiute-2020 - notebook v0

Validazione della pipeline per fasi: **raw → clean → mart**.

- scopo: verificare che ogni layer sia corretto e coerente con il precedente
- fonte: MIT — Opere pubbliche incompiute al 31/12/2020
- **non** sostituisce l'analisi pubblica


In [ ]:
import re
import yaml
import pandas as pd
from pathlib import Path

# --- Unici parametri da impostare manualmente ---
METRICA       = "importo_complessivo_qe"  # colonna numerica principale nel mart
METRICA_CLEAN = "importo_complessivo_qe"  # colonna corrispondente nel clean

# --- Anchor: usa il path del notebook se disponibile (VSCode), altrimenti cwd ---
try:
    _start = Path(__vsc_ipynb_file__).resolve().parent  # VSCode Jupyter
except NameError:
    _start = Path.cwd().resolve()

# Walk up finché non troviamo dataset.yml
candidate_dir = None
for probe in [_start, *_start.parents]:
    if (probe / "dataset.yml").exists():
        candidate_dir = probe
        break

if candidate_dir is None:
    raise FileNotFoundError(f"dataset.yml non trovato risalendo da {_start}")

cfg = yaml.safe_load((candidate_dir / "dataset.yml").read_text())

SLUG       = cfg["dataset"]["name"]
ANNO_RUN   = cfg["dataset"]["years"][-1]
MART_TABLE = cfg["mart"]["tables"][0]["name"]
ENCODING   = cfg.get("clean", {}).get("read", {}).get("encoding", "utf-8")
DELIM      = cfg.get("clean", {}).get("read", {}).get("delim", ",")
HEADER     = cfg.get("clean", {}).get("read", {}).get("header", True)
SKIP       = cfg.get("clean", {}).get("read", {}).get("skip", 0)
DECIMAL    = cfg.get("clean", {}).get("read", {}).get("decimal", ",")

DI_ROOT   = (candidate_dir / cfg["root"]).resolve()
RAW_DIR   = DI_ROOT / "data" / "raw"   / SLUG / str(ANNO_RUN)
CLEAN_DIR = DI_ROOT / "data" / "clean" / SLUG / str(ANNO_RUN)
MART_DIR  = DI_ROOT / "data" / "mart"  / SLUG / str(ANNO_RUN)
SQL_DIR   = candidate_dir / "sql"

print(f"slug      : {SLUG}")
print(f"anno_run  : {ANNO_RUN}")
print(f"mart table: {MART_TABLE}")
print(f"encoding  : {ENCODING}  delim: {repr(DELIM)}  decimal: {repr(DECIMAL)}")
print(f"header    : {HEADER}  skip: {SKIP}")


slug      : mit_opere_incompiute_2020
anno_run  : 2020
mart table: mart_opere_snapshot
encoding  : utf-8  delim: ';'  decimal: ','
header    : True  skip: 0



## SQL di riferimento

Le SQL sono la fonte di verità per capire cosa deve contenere ogni layer.


In [ ]:
for sql_file in sorted(SQL_DIR.glob("*.sql")):
    print(f"{'='*60}")
    print(f"  {sql_file.name}")
    print(f"{'='*60}")
    print(sql_file.read_text())
    print()


  clean.sql
with base as (
    select
        trim("Graduatoria_pubblicata_da") as graduatoria_pubblicata_da,
        cast("Anno_di_riferimento" as integer) as anno_di_riferimento,
        nullif(trim("Codice_CUP"), '') as codice_cup,
        trim("Ambito_soggettivo") as ambito_soggettivo,
        trim("Denominazione_Stazione_appaltante") as stazione_appaltante,
        nullif(trim("Codice_fiscale"), '') as codice_fiscale,
        trim("Stato_opera(d.m. 42/2013, art. 1, c. 2)") as stato_opera,
        nullif(trim("Localizzazione_codice_ISTAT"), '') as localizzazione_codice_istat,
        nullif(trim("Localizzazione_codice_NUTS"), '') as localizzazione_codice_nuts,
        trim("Ambito_oggettivo") as ambito_oggettivo,
        trim("Titolo_opera_incompiuta") as titolo_opera_incompiuta,
        trim("Descrizione_intervento") as descrizione_intervento,
        cast(replace(replace(nullif(trim("Importo_complessivo_aggiornato_ultimo_quadro_economico"), ''), '.', ''), ',', '.') as double) as 

## 1. Raw

Verifica che il file raw esista e sia leggibile.


In [ ]:
import duckdb

raw_files = sorted(RAW_DIR.glob("*.csv")) + sorted(RAW_DIR.glob("*.parquet"))
if not raw_files:
    raise FileNotFoundError(f"Nessun file raw trovato in {RAW_DIR}")

raw_path = raw_files[0]
print(f"File: {raw_path.name}  ({raw_path.stat().st_size / 1024:.0f} KB)")

# Usa duckdb per leggere CSV con separatore corretto
con = duckdb.connect()
raw_full = con.query(f"SELECT * FROM '{raw_path}'").fetchdf()

N_RAW = len(raw_full)
print(f"Righe raw   : {N_RAW}")
print(f"Colonne raw : {len(raw_full.columns)} -> {list(raw_full.columns)[:5]}...")
raw_full.head(3)


File: mit_opere_incompiute_2020.csv  (201 KB)
Righe raw   : 409
Colonne raw : 27 -> ['Graduatoria_pubblicata_da', 'Anno_di_riferimento', 'Codice_CUP', 'Ambito_soggettivo', 'Denominazione_Stazione_appaltante']...



## 2. Clean

Verifica schema e null. Il file pulito non deve contenere righe TOT o duplicati su codice_cup.


In [ ]:
clean_files = sorted(CLEAN_DIR.glob("*.parquet"))
if not clean_files:
    raise FileNotFoundError(f"Clean non trovato in {CLEAN_DIR}. Eseguire: toolkit run clean")

clean = pd.read_parquet(clean_files[0])
N_CLEAN = len(clean)

print(f"Righe clean : {N_CLEAN}")
print(f"Colonne     : {list(clean.columns)}")
clean.head(3)


Righe clean : 405
Colonne     : ['graduatoria_pubblicata_da', 'anno_di_riferimento', 'codice_cup', 'ambito_soggettivo', 'stazione_appaltante', 'codice_fiscale', 'stato_opera', 'localizzazione_codice_istat', 'localizzazione_codice_nuts', 'ambito_oggettivo', 'titolo_opera_incompiuta', 'descrizione_intervento', 'importo_complessivo_qe', 'importo_lavori_qe', 'importo_lavori_sal', 'importo_oneri_ultimazione', 'perc_avanzamento_lavori', 'mancanza_fondi', 'cause_tecniche', 'sopravvenienza_norme', 'fallimento_liquidazione', 'mancato_interesse', 'opera_fruibile_collettivita', 'uso_ridimensionato_opera', 'infrastruttura_rete', 'discontinuita_rete', 'livello_sviluppo']



In [ ]:
# N_RAW conta le righe raw incl. header — il raw ha 1 riga header
dropped = N_RAW - N_CLEAN
dropped_pct = dropped / N_RAW * 100 if N_RAW > 0 else 0

print(f"Righe raw         : {N_RAW}")
print(f"Righe clean       : {N_CLEAN}")
print(f"Righe filtrate    : {dropped} ({dropped_pct:.1f}%)")
print()
if dropped > 0:
    print("-> Filtri attivi: riga TOT + dup su codice_cup (2 duplicati)")
else:
    print("-> Nessuna riga filtrata.")

# Verifica che TOT non sia presente
tot_count = (clean['codice_cup'] == 'TOT').sum() if 'codice_cup' in clean.columns else -1
print(f"\nRighe 'TOT' in clean: {tot_count} (atteso: 0)")

print("\nNull per colonna clean:")
null_counts = clean.isnull().sum()
print(null_counts[null_counts > 0].to_string() if null_counts.any() else "  nessuno")


Righe raw         : 409
Righe clean       : 405
Righe filtrate    : 4 (1.0%)

-> Filtri attivi: riga TOT + dup su codice_cup (2 duplicati)

Righe 'TOT' in clean: 0 (atteso: 0)

Null per colonna clean:
ambito_soggettivo              38
localizzazione_codice_nuts    169
importo_oneri_ultimazione      51
perc_avanzamento_lavori         1
mancanza_fondi                148
cause_tecniche                162
sopravvenienza_norme          240
fallimento_liquidazione       198
mancato_interesse             234
discontinuita_rete            298



## 3. Mart

Verifica unicità sulle chiavi del GROUP BY, null e consistenza dei totali con il clean.


In [ ]:
mart_path = MART_DIR / f"{MART_TABLE}.parquet"
if not mart_path.exists():
    raise FileNotFoundError(f"Mart non trovato: {mart_path}. Eseguire: toolkit run mart")

mart = pd.read_parquet(mart_path)
print(f"Righe mart  : {len(mart)}")
print(f"Colonne     : {list(mart.columns)}")
mart.head(3)


Righe mart  : 405
Colonne     : ['anno_di_riferimento', 'codice_cup', 'titolo_opera_incompiuta', 'stazione_appaltante', 'localizzazione_codice_istat', 'localizzazione_codice_nuts', 'ambito_oggettivo', 'stato_opera', 'livello_sviluppo', 'perc_avanzamento_lavori', 'importo_complessivo_qe', 'importo_lavori_qe', 'importo_lavori_sal', 'importo_oneri_ultimazione', 'flag_mancanza_fondi', 'flag_cause_tecniche', 'flag_sopravvenienza_norme', 'flag_fallimento_liquidazione', 'flag_mancato_interesse', 'flag_opera_fruibile', 'flag_uso_ridimensionato', 'flag_infrastruttura_rete', 'flag_discontinuita_rete']



In [ ]:
# Verifica unicita su codice_cup (chiave univoca del dataset)
dups = mart.duplicated(subset=['codice_cup']).sum()
print(f"Duplicati su codice_cup: {dups}")
assert dups == 0, f"FAIL: {dups} duplicati trovati"
print("OK: codice_cup unico nel mart.")


Duplicati su codice_cup: 0
OK: codice_cup unico nel mart.



In [ ]:
if "anno_di_riferimento" in mart.columns:
    anni = sorted(mart["anno_di_riferimento"].unique())
    print(f"Anni nel mart: {anni}")

null_mart = mart.isnull().sum()
print("\nNull per colonna mart:")
print(null_mart[null_mart > 0].to_string() if null_mart.any() else "  nessuno")


Anni nel mart: [np.int32(2020)]

Null per colonna mart:
localizzazione_codice_nuts    169
perc_avanzamento_lavori         1
importo_oneri_ultimazione      51



In [ ]:
if METRICA in mart.columns and METRICA_CLEAN in clean.columns:
    tot_m  = mart[METRICA].sum()
    tot_c  = clean[METRICA_CLEAN].sum()
    delta_pct = abs(tot_m - tot_c) / tot_c * 100 if tot_c != 0 else float("nan")
    print(f"Totale mart  ({METRICA})      : {tot_m:,.2f}")
    print(f"Totale clean ({METRICA_CLEAN}): {tot_c:,.2f}")
    print(f"Delta %: {delta_pct:.4f}%")
    print("OK: i totali coincidono." if delta_pct < 0.01 else f"SKIP: differenza {delta_pct:.2f}% attesa")


Totale mart  (importo_complessivo_qe)      : 2,730,145,785.68
Totale clean (importo_complessivo_qe): 2,730,145,785.68
Delta %: 0.0000%
OK: i totali coincidono.



In [ ]:
PERIMETRO = "MIT opere incompiute 2020 — opere pubbliche incompiute per regione, causa e stato"

print(f"Slug         : {SLUG}")
print(f"Anno run     : {ANNO_RUN}")
print(f"Tabella mart : {MART_TABLE}")
print(f"Metrica mart : {METRICA}")
print(f"Metrica clean: {METRICA_CLEAN}")
print(f"Perimetro    : {PERIMETRO}")


Slug         : mit_opere_incompiute_2020
Anno run     : 2020
Tabella mart : mart_opere_snapshot
Metrica mart : importo_complessivo_qe
Metrica clean: importo_complessivo_qe
Perimetro    : MIT opere incompiute 2020 — opere pubbliche incompiute per regione, causa e stato

